# Carleton et al. (2022)

Illinois county-level LOCA2 indicators. Keep this notebook and `loca2_common.py` in the same folder. Open Jupyter in that folder and run the cells from top to bottom. Each notebook uses its own kernel.

The output folder and filename convention match the original workflow. Existing files are skipped (`OVERWRITE = False`). SSP245 and SSP370 and the four original reporting periods are retained. County aggregation uses cosine-latitude area correction, with no population, LandScan, or crop weights.

## 1. Import shared functions

Required packages: numpy, xarray, dask, netCDF4, cftime, pandas, geopandas, regionmask, and tqdm. Use your existing Keeling environment.

In [1]:
from pathlib import Path
import numpy as np
import xarray as xr
import loca2_common as common
from loca2_common import (
    aggregate_to_illinois_counties,
    annual_sum,
    daily_mean_temperature,
    to_fahrenheit,
)

## 2. Paper settings

In [2]:
PAPER_NAME = 'Carleton et al. (2022)'
REQUIRED_VARIABLES = ('tasmin', 'tasmax', 'pr')
OVERWRITE = False

## 3. Check paths and available runs

This cell lists the workload and loads county boundaries. It does not write climate output.

In [3]:
output_directory = common.PAPER_OUTPUT_ROOT / common.paper_directory_name(PAPER_NAME)
jobs = common.discover_paper_runs(REQUIRED_VARIABLES)
existing = sum(
    (output_directory / (
        f"{common.safe_token(PAPER_NAME)}_"
        f"{common.safe_token(model)}_{common.safe_token(member)}_{scenario}.nc"
    )).exists()
    for model, member, scenario in jobs
)
print("Input root:", common.PAPER_INPUT_ROOT)
print("Output folder:", output_directory)
print("Eligible runs:", len(jobs))
print("Existing files to skip:", existing if not OVERWRITE else 0)
print("Runs to process:", len(jobs) - existing if not OVERWRITE else len(jobs))
if not jobs:
    raise FileNotFoundError("No eligible runs found. Check input paths and required variables.")
counties = common.load_counties()
print("Counties:", len(counties))

Input root: /data/keeling/a/tahsina2/b/Climate_Impact/Data/LOCA2_Illinois
Output folder: /data/keeling/a/tahsina2/b/Climate_Impact/Data/LOCA2_Illinois/temp/Carleton et al. (2022)
Eligible runs: 156
Existing files to skip: 0
Runs to process: 156
Counties: 102


## 4. Paper indicator calculation

In [4]:
def calculate_carleton(inputs):
    """
    Calculate annual Carleton et al. weather indicators at each
    LOCA2 grid cell and aggregate them to Illinois counties.

    Output dimensions:
        year × county
    """
    required_variables = {'tasmin', 'tasmax', 'pr'}
    missing_variables = required_variables - set(inputs)
    if missing_variables:
        raise KeyError(f'Missing required Carleton input variables: {sorted(missing_variables)}')
    original_time_sizes = {'tasmin': inputs['tasmin'].sizes['time'], 'tasmax': inputs['tasmax'].sizes['time'], 'pr': inputs['pr'].sizes['time']}
    tasmin_c, tasmax_c, precipitation_in = xr.align(inputs['tasmin'], inputs['tasmax'], inputs['pr'], join='inner', copy=False)
    aligned_time_size = tasmin_c.sizes['time']
    if aligned_time_size == 0:
        raise ValueError('No shared dates were found among tasmin, tasmax, and precipitation.')
    removed_dates = {variable_name: original_size - aligned_time_size for variable_name, original_size in original_time_sizes.items()}
    if any((count > 0 for count in removed_dates.values())):
        print(f'Time alignment retained shared dates only. Removed dates: {removed_dates}')
    if not (tasmin_c.indexes['time'].equals(tasmax_c.indexes['time']) and tasmin_c.indexes['time'].equals(precipitation_in.indexes['time'])):
        raise ValueError('Time indexes remain inconsistent after alignment.')
    tmean_c = daily_mean_temperature(tasmin_c, tasmax_c)
    tmean_f = to_fahrenheit(tmean_c).rename('daily_mean_temperature_f')
    tmean_f.attrs.update(units='degree_Fahrenheit', long_name='daily mean temperature', daily_calculation='(tasmin + tasmax) / 2')
    precipitation_in = precipitation_in.rename('daily_precipitation')
    precipitation_units = str(precipitation_in.attrs.get('units', '')).strip().lower().replace('−', '-')
    accepted_precipitation_units = {'inch', 'inches', 'in', 'inch/day', 'inches/day', 'in/day', 'inch day-1', 'inches day-1', 'in day-1', 'inch d-1', 'inches d-1', 'in d-1'}
    if precipitation_units not in accepted_precipitation_units:
        raise ValueError(f"Carleton precipitation must represent daily precipitation in inches. Found units: {precipitation_in.attrs.get('units')!r}")
    precipitation_in.attrs.update(units='inch day-1', long_name='daily precipitation')
    variables = {'annual_sum_temperature': annual_sum(tmean_f), 'annual_sum_temperature_squared': annual_sum(tmean_f ** 2), 'annual_sum_temperature_cubed': annual_sum(tmean_f ** 3), 'annual_sum_temperature_fourth_power': annual_sum(tmean_f ** 4), 'annual_precipitation': annual_sum(precipitation_in), 'annual_sum_precipitation_squared': annual_sum(precipitation_in ** 2)}
    result = xr.Dataset(variables)
    result['annual_sum_temperature'].attrs.update(units='degree_Fahrenheit day', long_name='annual sum of daily mean temperature', formula='sum(T)', temperature_statistic='daily mean temperature calculated as (tasmin + tasmax) / 2')
    result['annual_sum_temperature_squared'].attrs.update(units='degree_Fahrenheit^2 day', long_name='annual sum of squared daily mean temperature', formula='sum(T^2)')
    result['annual_sum_temperature_cubed'].attrs.update(units='degree_Fahrenheit^3 day', long_name='annual sum of cubed daily mean temperature', formula='sum(T^3)')
    result['annual_sum_temperature_fourth_power'].attrs.update(units='degree_Fahrenheit^4 day', long_name='annual sum of daily mean temperature raised to the fourth power', formula='sum(T^4)')
    result['annual_precipitation'].attrs.update(units='inch', long_name='annual total precipitation', formula='sum(P)', temporal_aggregation='sum of daily precipitation')
    result['annual_sum_precipitation_squared'].attrs.update(units='inch^2', long_name='annual sum of squared daily precipitation', formula='sum(P^2)', calculation_note='daily precipitation is squared before annual summation; this is not the square of annual precipitation')
    result.attrs.update(paper='Carleton et al. (2022)', temporal_resolution='annual', temperature_units='degrees Fahrenheit', precipitation_units='inches', temperature_statistic='daily mean calculated as (tasmin + tasmax) / 2', indicators='annual sums of T, T^2, T^3, T^4, P, and P^2', time_alignment='shared dates among tasmin, tasmax, and precipitation')
    result = aggregate_to_illinois_counties(dataset=result, counties=counties)
    result.attrs.update(spatial_resolution='Illinois county', weighting='grid-cell area correction only; no LandScan, population, or crop weighting')
    return result

## 5. Generate output

Existing matching filenames are skipped. A skipped file is not recalculated or checked for completeness.

In [ ]:
common.run_paper(
    paper_name=PAPER_NAME,
    required_variables=REQUIRED_VARIABLES,
    calculation_function=calculate_carleton,
    overwrite=OVERWRITE,
)

Carleton et al. (2022):   0%|          | 0/156 [00:00<?, ?run/s]

## 6. Inspect an output file

In [ ]:
files = sorted(output_directory.glob(f"{common.safe_token(PAPER_NAME)}_*.nc"))
if files:
    with xr.open_dataset(files[0], decode_timedelta=False) as dataset:
        print("Example:", files[0].name)
        print("Dimensions:", dict(dataset.sizes))
        print("Variables:", list(dataset.data_vars))
        print("Spatial resolution:", dataset.attrs.get("spatial_resolution"))
        assert dataset.sizes.get("county") == 102, "Output is not a 102-county dataset."
else:
    print("No output files found. Review the processing messages above.")